[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: variables y secretos de Actions

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

Este cuaderno resuelve la actividad de `06_expresiones_y_variables.ipynb`. Usa el repositorio `demo-github-actions` creado en el cuaderno 02.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Definir una variable de repositorio

En el repositorio de la actividad previa, abrir `Settings` → `Secrets and variables` → `Actions` → pestaña `Variables` y crear la variable `variable_demo` con el valor `Hola, Mundo.`. Observar que *GitHub* cambia el nombre a mayúsculas.

Las reglas de nombres de variables y secretos de *GitHub* son: solo letras, números y guion bajo; no empiezan por un número; no empiezan por `GITHUB_`; y no distinguen mayúsculas de minúsculas (se guardan en mayúsculas). La siguiente función las implementa para comprobar un nombre antes de enviarlo.

La siguiente celda definirá la función `normalizar_nombre(nombre)`, que valida un nombre con una expresión regular y lo devuelve en mayúsculas. Si el nombre no cumple las reglas, lanza `ValueError`.

In [ ]:
import re

def normalizar_nombre(nombre):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", nombre):
        raise ValueError(f"Nombre no válido: {nombre!r}")
    if nombre.upper().startswith("GITHUB_"):
        raise ValueError("Los nombres no pueden empezar por GITHUB_")
    return nombre.upper()

La siguiente celda probará la función con el nombre de la actividad y con tres nombres incorrectos. El primero debe devolver `VARIABLE_DEMO`; los otros tres deben lanzar `ValueError`.

In [ ]:
assert normalizar_nombre("variable_demo") == "VARIABLE_DEMO"
assert normalizar_nombre("MUY_SECRETO") == "MUY_SECRETO"
for incorrecto in ("1_variable", "github_token", "con espacio"):
    try:
        normalizar_nombre(incorrecto)
    except ValueError as error:
        print("Rechazado:", error)
    else:
        raise AssertionError(incorrecto)
print(normalizar_nombre("variable_demo"))

La siguiente celda creará la variable del repositorio con el *CLI*. El nombre se pasa ya normalizado y el valor viene de la variable de entorno `VALOR_VARIABLE`, definida en la misma celda con el texto de la actividad. Después listará las variables del repositorio.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/demo-github-actions"
VALOR_VARIABLE="Hola, Mundo."
gh variable set VARIABLE_DEMO --body "${VALOR_VARIABLE}"
gh variable list

**Salida esperada.** `gh variable list` muestra una fila con el nombre `VARIABLE_DEMO`, el valor `Hola, Mundo.` y la fecha de actualización. El nombre aparece en mayúsculas aunque se haya escrito en minúsculas.

## Ejercicio 2. Definir un secreto de repositorio

En la pestaña `Secrets`, crear el secreto `MUY_SECRETO` con el valor `Esto no volverá a ser desplegado`. Comprobar que, una vez guardado, el valor ya no se puede ver.

Un secreto se escribe sin pasar por un archivo ni por el historial del terminal: `gh secret set` lee el valor de la entrada estándar. Aquí el valor llega desde una variable de entorno.

La siguiente celda guardará el secreto `MUY_SECRETO` leyendo su valor de la entrada estándar y luego listará los secretos. La lista contiene nombres y fechas, nunca valores.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/demo-github-actions"
VALOR_SECRETO="Esto no volverá a ser desplegado"
printf '%s' "${VALOR_SECRETO}" | gh secret set MUY_SECRETO
gh secret list

**Salida esperada.** `gh secret list` presenta `MUY_SECRETO` con la fecha de actualización. No existe ningún comando que devuelva el valor; para cambiarlo hay que volver a escribirlo.

## Ejercicio 3. Resumir los contextos y su sintaxis (complemento)

El cuaderno 06 afirma que una variable de entorno se escribe `$<contexto>.<nombre>`. Esa sintaxis no existe tal cual: indicar cómo se escribe cada caso correctamente.

*Nota para quien imparte el curso: este ejercicio no está en el cuaderno original; se añade para corregir una imprecisión.*

Hay dos sintaxis distintas. Dentro de un `run`, el *shell* lee una variable de entorno con `$NOMBRE`. En el resto del *YAML* se usan expresiones `${{ contexto.nombre }}`, que *GitHub* evalúa **antes** de entregar el texto al *shell*.

La siguiente celda mostrará con `sustituir_expresiones()` cómo se evalúan las expresiones de los contextos `env`, `vars` y `secrets`, y comprobará que `$NOMBRE` no se toca (lo resuelve el *shell*, no *GitHub*).

In [ ]:
contextos = {
    "env": {"VARIABLE_LOCAL": "valor local"},
    "vars": {"VARIABLE_DEMO": "Hola, Mundo."},
    "secrets": {"MUY_SECRETO": "valor-de-demostracion"},
}
for texto in ("echo ${{ env.VARIABLE_LOCAL }}", "echo ${{ vars.VARIABLE_DEMO }}", "echo $VARIABLE_LOCAL"):
    print(f"{texto!r:40} -> {u.sustituir_expresiones(texto, contextos)!r}")

assert u.sustituir_expresiones("echo ${{ vars.VARIABLE_DEMO }}", contextos) == "echo Hola, Mundo."
assert u.sustituir_expresiones("echo $VARIABLE_LOCAL", contextos) == "echo $VARIABLE_LOCAL"
assert u.sustituir_expresiones("echo ${{ vars.NO_EXISTE }}", contextos) == "echo "

Una expresión con un nombre que no existe se evalúa como cadena vacía, sin error. Por eso conviene verificar que el nombre esté bien escrito (y en mayúsculas, como lo guarda *GitHub*).

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>